# PCA Analysis Across Strains - Cleaned Version

This notebook performs Principal Component Analysis (PCA) on barcode count data across four bacterial strains (ECI, ECJ, ST69, ST73) under different treatment conditions.

## Features:
- Consistent data processing pipeline
- Standardized visualizations with consistent colors/symbols
- Functionalized code to reduce duplication
- Robust error handling and data validation

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from functools import reduce
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Configuration and Constants

In [ ]:

STRAIN_FILES = {
    'ECI': {
        'LCM': '../data/TV200_ECI_LCM_filtered.csv',
        'Pretreated': '../data/TV238_ECI_Pretreated_filtered.csv'
    },
    'ECJ': {
        'LCM': '../data/TV182_ECJ_LCM_filtered.csv',
        'Pretreated': '../data/TV240_ECJ_Pretreated_filtered.csv',
        'GermFree': '../data/TV248_ECJ_GermFree_filtered.csv'
    },
    'ST69': {
        'LCM': '../data/TV197_ST69_LCM_filtered.csv',
        'Pretreated': '../data/TV239_ST69_Pretreated_filtered.csv'
    },
    'ST73': {
        'LCM': '../data/TV199_ST73_LCM_filtered.csv',
        'Pretreated': '../data/TV241_ST73_Pretreated_filtered.csv',
        'GermFree': '../data/TV249_ST73_GermFree_filtered.csv'
    }
}

# Experiment ID to treatment mapping
TREATMENT_MAP = {
    'TV182': 'LCM',
    'TV197': 'LCM', 
    'TV199': 'LCM',
    'TV200': 'LCM',
    'TV238': 'Pretreated',
    'TV239': 'Pretreated', 
    'TV240': 'Pretreated',
    'TV241': 'Pretreated',
    'TV248': 'GermFree',
    'TV249': 'GermFree'
}

# Consistent color and symbol mapping
COLOR_MAP = {
    'D1': '#1f77b4',      # Blue
    'D2': '#ff7f0e',      # Orange  
    'D3': '#2ca02c',      # Green
    'D4': '#d62728',      # Red
    'D8': '#9467bd',      # Purple
    'inoculum': '#17becf' # Cyan
}

TREATMENT_COLORS = {'LCM': "#EE7457",
                       'Pretreated': "#104E8B",
                       'GermFree': "#87CEFF",
                       'Inoculum': "#404040"
                      }


SYMBOL_MAP = {
    'LCM': 'circle',
    'Pretreated': 'square', 
    'GermFree': 'diamond'
}

print("Configuration loaded successfully")
print(f"Strains to analyze: {list(STRAIN_FILES.keys())}")

## Data Processing Functions

In [ ]:
def load_strain_data(strain_name, file_mapping):
    """Load and merge all CSV files for a given strain.
    
    Args:
        strain_name (str): Name of the strain (e.g., 'ECI', 'ECJ')
        file_mapping (dict): Dictionary mapping treatment -> filename
        
    Returns:
        pd.DataFrame: Merged DataFrame with barcode and new_locus_tag as index
    """
    print(f"\nLoading data for {strain_name} strain...")
    
    # Find all files for this strain
    files = []
    for treatment, filename in file_mapping.items():
        if Path(filename).exists():
            files.append(Path(filename))
            print(f"  Found: {filename} ({treatment})")
        else:
            print(f"  WARNING: Missing file {filename} for {treatment}")
    
    if not files:
        raise FileNotFoundError(f"No files found for strain {strain_name}")
    
    # Load and merge DataFrames
    dfs = []
    for file in files:
        df = pd.read_csv(file)
        print(f"  {file.name}: {len(df)} barcodes, {df.shape[1]-2} samples")
        dfs.append(df)
    
    # Merge all DataFrames on barcode and new_locus_tag
    merged_df = reduce(
        lambda left, right: pd.merge(left, right, on=['barcode', 'new_locus_tag'], how='outer'),
        dfs
    )
    
    print(f"  Merged dataset: {len(merged_df)} total barcodes")
    return merged_df.set_index(['barcode', 'new_locus_tag'])

In [ ]:
def parse_sample_metadata(df, strain_name):
    """Parse sample metadata from column names.
    
    Args:
        df (pd.DataFrame): DataFrame with sample columns
        strain_name (str): Strain name for validation
        
    Returns:
        pd.DataFrame: Sample metadata with columns [exp, strain, mouse, day, treatment]
    """
    sample_ids = df.columns.tolist()
    
    metadata_rows = []
    for sample_id in sample_ids:
        # Skip non-sample columns
        if not ('_' in sample_id and 'usingintergenic' in sample_id.lower()):
            continue
            
        parts = sample_id.split('_')
        
        # Handle different column name formats
        if len(parts) >= 4:
            exp_id = parts[0]
            
            # Handle strain in column name
            if strain_name in sample_id:
                strain_part = strain_name
                mouse_idx = 2
            else:
                strain_part = strain_name  # Use provided strain
                mouse_idx = 1
            
            if mouse_idx < len(parts):
                mouse = parts[mouse_idx]
                day_part = parts[mouse_idx + 1] if mouse_idx + 1 < len(parts) else 'D1'
            else:
                mouse = 'unknown'
                day_part = 'D1'
            
            # Standardize day labels
            if 'inoculum' in day_part or day_part in ['1', '2'] or 'inoculum' in mouse:
                day = 'inoculum'
                
            else:
                day = day_part
            
            # Map treatment
            treatment = TREATMENT_MAP.get(exp_id, 'Unknown')
            
            metadata_rows.append({
                'sample_id': sample_id,
                'exp': exp_id,
                'strain': strain_part, 
                'mouse': mouse,
                'day': day,
                'treatment': treatment
            })
    
    metadata_df = pd.DataFrame(metadata_rows).set_index('sample_id')
    metadata_df['treatment'] = (metadata_df['treatment'] + " " + metadata_df['day']).str.split(" D", expand=True)[0]
    metadata_df['day'] = metadata_df['day'].replace({'inoculum': 'D0'})
    print(f"  Parsed {len(metadata_df)} samples")
    print(f"  Treatments: {metadata_df['treatment'].value_counts().to_dict()}")
    print(f"  Days: {metadata_df['day'].value_counts().to_dict()}")

    return metadata_df

In [ ]:
def normalize_counts(df, method='log2_cpm', pseudocount=0.5):
    """Normalize count data consistently.
    
    Args:
        df (pd.DataFrame): Count data with samples as columns
        method (str): Normalization method ('log2_cpm', 'clr')
        pseudocount (float): Pseudocount for log transformation
        
    Returns:
        pd.DataFrame: Normalized data
    """
    print(f"  Applying {method} normalization...")
    
    if method == 'log2_cpm':
        # Convert to counts per million and log2 transform
        cpm = df.divide(df.sum(axis=0), axis=1) * 1e6
        normalized = np.log2(cpm + pseudocount)
    elif method == 'clr':
        # Centered log-ratio transformation
        df_pseudo = df + pseudocount
        log_data = np.log(df_pseudo)
        geometric_means = np.exp(log_data.mean(axis=0))
        normalized = log_data.subtract(np.log(geometric_means), axis=1)
    else:
        raise ValueError(f"Unknown normalization method: {method}")
    
    # Remove rows with any NaN values
    normalized_clean = normalized.dropna()
    n_removed = len(normalized) - len(normalized_clean)
    if n_removed > 0:
        print(f"  Removed {n_removed} barcodes with missing values")
   
    print(f"  Final dataset: {normalized_clean.shape[0]} barcodes × {normalized_clean.shape[1]} samples")
    return normalized_clean

In [ ]:
def perform_pca(data, n_components=2, rev_pc1=False,
               rev_pc2=False):
    """Perform PCA on normalized data.
    
    Args:
        data (pd.DataFrame): Normalized count data
        n_components (int): Number of principal components
        
    Returns:
        tuple: (pca_results_df, pca_object)
    """
    print(f"  Performing PCA with {n_components} components...")
    
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(data.T)  # Transpose: samples as rows
    
    # Create results DataFrame
    pc_columns = [f'PC{i+1}' for i in range(n_components)]
    pca_df = pd.DataFrame(
        pca_result, 
        index=data.columns,  # Sample names
        columns=pc_columns
    )
    if rev_pc1:
        pca_df['PC1'] = pca_df['PC1']*-1
    if rev_pc2:
        pca_df['PC2'] = pca_df['PC2']*-1
    
    # Print variance explained
    var_explained = pca.explained_variance_ratio_
    for i, var in enumerate(var_explained):
        print(f"    PC{i+1}: {var:.1%} variance explained")
    print(f"    Total: {sum(var_explained):.1%} variance explained")
    
    return pca_df, pca

## Visualization Functions

In [ ]:
def create_pca_plot_original(pca_data, metadata, strain_name, pca_obj=None, title_suffix=""):
    """Create a standardized PCA plot.
    
    Args:
        pca_data (pd.DataFrame): PCA results with PC1, PC2 columns
        metadata (pd.DataFrame): Sample metadata
        strain_name (str): Strain name for the title
        pca_obj: PCA object for variance explained
        title_suffix (str): Additional title text
        
    Returns:
        plotly.graph_objects.Figure: PCA plot
    """
    # Merge PCA data with metadata
    plot_data = pca_data.join(metadata, how='inner')
    
    if plot_data.empty:
        print(f"Warning: No data to plot for {strain_name}")
        return go.Figure()
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each treatment-day combination
    for treatment in plot_data['treatment'].unique():
        if pd.isna(treatment):
            continue
            
        treatment_data = plot_data[plot_data['treatment'] == treatment]
        
        for day in treatment_data['day'].unique():
            if pd.isna(day):
                continue
                
            day_data = treatment_data[treatment_data['day'] == day]
            
            # Get consistent colors and symbols
            color = COLOR_MAP.get(day, '#808080')  # Gray for unknown
            symbol = SYMBOL_MAP.get(treatment, 'circle')
            
            fig.add_trace(go.Scatter(
                x=day_data['PC1'],
                y=day_data['PC2'],
                mode='markers',
                marker=dict(
                    color=color,
                    symbol=symbol,
                    size=12,
                    line=dict(width=2, color='black')
                ),
                name=f'{treatment} - {day}',
                text=[f'Sample: {idx}<br>Treatment: {treatment}<br>Day: {day}<br>Mouse: {row["mouse"]}' 
                      for idx, row in day_data.iterrows()],
                hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
            ))
    
    # Calculate axis labels with variance explained
    if pca_obj is not None:
        var_explained = pca_obj.explained_variance_ratio_
        pc1_label = f"PC1 ({var_explained[0]:.1%})"
        pc2_label = f"PC2 ({var_explained[1]:.1%})"
    else:
        pc1_label = "PC1"
        pc2_label = "PC2"
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f'PCA Analysis - {strain_name} Strain{title_suffix}',
            x=0.5,
            font=dict(size=16, family="Arial Black")
        ),
        xaxis=dict(
            title=pc1_label,
            gridcolor='lightgray',
            linecolor='black',
            linewidth=1,
            mirror=True
        ),
        yaxis=dict(
            title=pc2_label,
            gridcolor='lightgray', 
            linecolor='black',
            linewidth=1,
            mirror=True
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            x=1.02,
            y=1,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1
        ),
        width=900,
        height=600
    )
    
    return fig

In [ ]:
# ── Color & gradient setup ────────────────────────────────────────────────────

TREATMENT_COLORS = {'LCM': "#EE7457",
                       'Pretreated': "#104E8B",
                       'GermFree': "#87CEFF",
            
                    'LCM inoculum': "#555555",
                    'Pretreated inoculum': '#111111',
                    'GermFree inoculum': '#888888'
                      }


TREATMENT_LABELS = {
    "LCM": "With microbiota (LCM)",
    "Pretreated": "Without microbiota (treated)",
    'GermFree': "Without microbiota (GF)",
    'LCM inoculum': "LCM inoculum",
    'Pretreated inoculum': "treated inoculum",
    'GermFree inoculum': "GF inoculum"
}

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

def make_gradient(base_hex, days, lightest=(230, 230, 230)):
    """D1 = darkest (base color), last day = lightest."""
    base_rgb = hex_to_rgb(base_hex)
    n = len(days)
    colors = {}
    for i, day in enumerate(days):
        t = i / (n - 1) if n > 1 else 0
        r = int(base_rgb[0] + t * (lightest[0] - base_rgb[0]))
        g = int(base_rgb[1] + t * (lightest[1] - base_rgb[1]))
        b = int(base_rgb[2] + t * (lightest[2] - base_rgb[2]))
        colors[day] = f"#{r:02X}{g:02X}{b:02X}"
    return colors


# ── Plot function ─────────────────────────────────────────────────────────────

def create_pca_plot(pca_data, metadata, strain_name, pca_obj=None, 
                    title_suffix="", mode="simple",
                   w=700, h=600):
    """Create a standardized PCA plot.
    
    Args:
        pca_data (pd.DataFrame): PCA results with PC1, PC2 columns
        metadata (pd.DataFrame): Sample metadata
        strain_name (str): Strain name for the title
        pca_obj: PCA object for variance explained
        title_suffix (str): Additional title text
        mode (str): 'simple' — one color per treatment, same marker;
                    'complex' — gradient per treatment x day, circle markers
        
    Returns:
        plotly.graph_objects.Figure: PCA plot
    """
    plot_data = pca_data.join(metadata, how='inner')
    
    if plot_data.empty:
        print(f"Warning: No data to plot for {strain_name}")
        return go.Figure()

    fig = go.Figure()
    
    # Sort days so gradient goes D1 → D8
    all_days = sorted(plot_data['day'].dropna().unique())

    if mode == "simple":
        for treatment in plot_data['treatment'].unique():
            if pd.isna(treatment):
                continue
            t_data = plot_data[plot_data['treatment'] == treatment]
            color = TREATMENT_COLORS.get(treatment, '#808080')
            
            fig.add_trace(go.Scatter(
                x=t_data['PC1'],
                y=t_data['PC2'],
                mode='markers',
                marker=dict(
                    color=color,
                    symbol='circle',
                    size=16,
                    line=dict(width=2, color='black')
                ),
                name=TREATMENT_LABELS.get(treatment, treatment),
                text=[f'Sample: {idx}<br>Treatment: {treatment}<br>Day: {row["day"]}<br>Mouse: {row["mouse"]}'
                      for idx, row in t_data.iterrows()],
                hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
            ))

    elif mode == "complex":
        for treatment in plot_data['treatment'].unique():
            if pd.isna(treatment):
                continue
            base_color = TREATMENT_COLORS.get(treatment, '#808080')
            gradient = make_gradient(base_color, all_days)
            t_data = plot_data[plot_data['treatment'] == treatment]

            for day in all_days:
                day_data = t_data[t_data['day'] == day]
                if day_data.empty:
                    continue

                fig.add_trace(go.Scatter(
                    x=day_data['PC1'],
                    y=day_data['PC2'],
                    mode='markers',
                    marker=dict(
                        color=gradient[day],
                        symbol='circle',
                        size=14,
                        line=dict(width=2, color='black')
                    ),
                    name=f'{TREATMENT_LABELS.get(treatment, treatment)} – {day}',
                    text=[f'Sample: {idx}<br>Treatment: {treatment}<br>Day: {day}<br>Mouse: {row["mouse"]}'
                          for idx, row in day_data.iterrows()],
                    hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
                ))
    
    # Axis labels
    if pca_obj is not None:
        var_explained = pca_obj.explained_variance_ratio_
        pc1_label = f"PC1 ({var_explained[0]:.1%})"
        pc2_label = f"PC2 ({var_explained[1]:.1%})"
    else:
        pc1_label, pc2_label = "PC1", "PC2"
    
    fig.update_layout(
        title=dict(
            text=f'PCA Analysis - {strain_name} Strain{title_suffix}',
            x=0.5,
            font=dict(size=16, family="Arial Black")
        ),
        xaxis=dict(title=dict(text=pc1_label, font=dict(size=18)), 
                   gridcolor='lightgray', tickfont=dict(size=18),
                   linecolor='black', linewidth=1, mirror=True),
        yaxis=dict(title=dict(text=pc2_label,font=dict(size=18)),
                   tickfont=dict(size=18),
                   gridcolor='lightgray', 
                   linecolor='black', linewidth=1, mirror=True),
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(x=1.02, y=1, bgcolor='rgba(255,255,255,0.8)', bordercolor='black', borderwidth=1),
        width=w,
        height=h,
        template='simple_white'
    )
    
    return fig

In [ ]:
def save_plot(fig, filename, output_dir='.', formats=['png'], w=700, h=600):
    """Save plot in multiple formats.
    
    Args:
        fig: Plotly figure
        filename (str): Base filename without extension
        formats (list): List of formats to save ['png', 'svg', 'pdf']
    """
    for fmt in formats:
        if fmt == 'png':
            fig.write_image(f"{output_dir}/{filename}.png", width=w, height=h, scale=2)
            print(f"  Saved: {output_dir}/{filename}.png")
        elif fmt == 'svg':
            fig.write_html(f"{output_dir}/{filename}.svg")
            print(f"  Saved: {output_dir}/{filename}.svg")
        elif fmt == 'pdf':
            fig.write_image(f"{output_dir}/{filename}.pdf", width=w, height=h, scale=2)
            print(f"  Saved: {output_dir}/{filename}.pdf")

## Main Analysis Pipeline

In [ ]:
def analyze_strain(strain_name, file_mapping, 
                   save_plots=True, days=[], 
                   normalization='log2_cpm', 
                   mode='simple',
                   rev_pc1=False,
                   rev_pc2=False,
                   w=700, h=600):
    """Complete analysis pipeline for a single strain.
    
    Args:
        strain_name (str): Name of the strain
        file_mapping (dict): Dictionary mapping treatment -> filename
        save_plots (bool): Whether to save plot files
        normalization (str): Normalization method to use
        
    Returns:
        tuple: (pca_data, metadata, figure)
    """
    print(f"\n{'='*50}")
    print(f"ANALYZING {strain_name} STRAIN")
    print(f"{'='*50}")
    
    try:
        # 1. Load and merge data
        raw_data = load_strain_data(strain_name, file_mapping)
        
        # 2. Parse sample metadata
        metadata = parse_sample_metadata(raw_data, strain_name)
        
        if len(days) > 0:
            metadata = metadata[metadata['day'].isin(days)]
            raw_data = raw_data[metadata.index]
        print(f"Days analyzed: {metadata['day'].unique()}")
        # 3. Normalize count data
        normalized_data = normalize_counts(raw_data, method=normalization)
        
        # 4. Perform PCA
        pca_data, pca_obj = perform_pca(normalized_data, 
                                       rev_pc1=rev_pc1,
                                       rev_pc2=rev_pc2)
        
        # 5. Create visualization
        print(f"  Creating visualization...")
        fig = create_pca_plot(pca_data, metadata, strain_name, pca_obj, mode=mode, w=w, h=h)
        
        
        # 6. Save plots if requested
        if save_plots:
            prefix = "_" + "_".join(days) if len(days) > 0 else ''
            date_str = datetime.now().strftime('%Y-%m-%d')
            filename = f"{date_str}_PCA_{strain_name}_cleaned{prefix}"
            save_plot(fig, filename, output_dir = "../figures", formats=['png'], w=w, h=h)
        
        print(f"\n✅ {strain_name} analysis completed successfully!")
        return pca_data, metadata, fig
        
    except Exception as e:
        print(f"\n❌ Error analyzing {strain_name}: {str(e)}")
        return None, None, None

## Run Analysis for All Strains

In [ ]:
# Store results for all strains
results = {}

# Analyze each strain
for strain, files in STRAIN_FILES.items():
    if strain =='ST69':
        rev_pc1=True
        rev_pc2=False
    elif strain == 'ECI':
        rev_pc1=True
        rev_pc2=True
    else:
        rev_pc1=False
        rev_pc2=False
    pca_data, metadata, fig = analyze_strain(strain, files, mode='complex',
                                             rev_pc1=rev_pc1,
                                             rev_pc2=rev_pc2,
                                             w=800, h=600,
                                             save_plots=True,)
    fig.update_layout(template='simple_white')
    if pca_data is not None:
        results[strain] = {
            'pca_data': pca_data,
            'metadata': metadata,
            'figure': fig
        }
        
        # Display the plot
        print(f"\nDisplaying {strain} PCA plot:")
        fig.show()
    else:
        print(f"\nSkipping {strain} due to analysis failure")

print(f"\n\n{'='*60}")
print(f"ANALYSIS SUMMARY")
print(f"{'='*60}")
print(f"Successfully analyzed: {list(results.keys())}")
print(f"Failed: {[s for s in STRAIN_FILES.keys() if s not in results]}")

## Different days for different strains

In [ ]:
results = {}

In [ ]:
strain = 'ECI'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain],
                                         days = ['D1', 'D0'], 
                                         save_plots=True,
                                         w=800, h=600)
if pca_data is not None:
    results[strain] = {
        'pca_data': pca_data,
        'metadata': metadata,
        'figure': fig
    }

    # Display the plot
    print(f"\nDisplaying {strain} PCA plot:")
    fig.show()
else:
    print(f"\nSkipping {strain} due to analysis failure")

In [ ]:
strain = 'ECJ'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain],
                                         days = ['D2', 'D0'],
                                         rev_pc1=True,
                                         save_plots=True,
                                         w=800, h=600)
if pca_data is not None:
    results[strain] = {
        'pca_data': pca_data,
        'metadata': metadata,
        'figure': fig
    }
    
    # Display the plot
    print(f"\nDisplaying {strain} PCA plot:")
    fig.show()
else:
    print(f"\nSkipping {strain} due to analysis failure")

In [ ]:
strain = 'ST69'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain],
                                         days = ['D2', 'D0'],
                                         rev_pc1=True,
                                         save_plots=True,
                                         w=800, h=600)
fig.update_layout(template='simple_white')
if pca_data is not None:
    results[strain] = {
        'pca_data': pca_data,
        'metadata': metadata,
        'figure': fig
    }
    
    # Display the plot
    print(f"\nDisplaying {strain} PCA plot:")
    fig.show()
else:
    print(f"\nSkipping {strain} due to analysis failure")

In [ ]:
strain = 'ST73'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain],
                                         days = ['D1', 'D0'], 
                                         
                                         save_plots=True,
                                         w=800, h=600)
fig.update_layout(template='simple_white')
if pca_data is not None:
    results[strain] = {
        'pca_data': pca_data,
        'metadata': metadata,
        'figure': fig
    }
    
    # Display the plot
    print(f"\nDisplaying {strain} PCA plot:")
    fig.show()
else:
    print(f"\nSkipping {strain} due to analysis failure")

# The end

In [ ]:
EcST131 I - D1
EcST131 J - D2
EcST69 - D2
EcST73 - D1



In [ ]:
strain = 'ECI'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain], mode='complex',
                                          save_plots=True, w=900, h=600)
fig.show()

In [ ]:
strain = 'ECJ'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain], mode='complex',
                                          save_plots=True, w=900, h=600)
fig.show()

In [ ]:
strain = 'ST69'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain], mode='complex',
                                          save_plots=True, w=900, h=600)
fig.show()

In [ ]:
strain = 'ST73'
pca_data, metadata, fig = analyze_strain(strain, STRAIN_FILES[strain], mode='complex',
                                          save_plots=True, w=900, h=600)
fig.show()

## Summary and Data Quality Report

In [ ]:
# Generate summary report
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

for strain, data in results.items():
    metadata = data['metadata']
    pca_data = data['pca_data']
    
    print(f"\n{strain} Strain:")
    print(f"  Samples analyzed: {len(metadata)}")
    print(f"  Treatments: {', '.join(metadata['treatment'].unique())}")
    print(f"  Time points: {', '.join(sorted(metadata['day'].unique()))}")
    print(f"  Sample distribution:")
    
    for treatment in metadata['treatment'].unique():
        treatment_data = metadata[metadata['treatment'] == treatment]
        day_counts = treatment_data['day'].value_counts().sort_index()
        print(f"    {treatment}: {dict(day_counts)}")

print("\n" + "="*60)
print("FILES GENERATED")
print("="*60)

date_str = datetime.now().strftime('%Y-%m-%d')

for strain in results.keys():
    print(f"  📊 {date_str}_PCA_{strain}_cleaned.png")

print(f"\n✅ Analysis pipeline completed successfully!")
print(f"📓 Clean, reproducible notebook: PCA_analysis_cleaned.ipynb")

## Looking for winners

In [ ]:
strain_name = 'ST69'
# 1. Load and merge data
raw_data = load_strain_data(strain_name, STRAIN_FILES[strain_name])

# 2. Parse sample metadata
metadata = parse_sample_metadata(raw_data, strain_name)

# 3. Normalize count data
normalized_data = normalize_counts(raw_data)

In [ ]:
normalized_data.shape

In [ ]:
test = (normalized_data[normalized_data.TV197_AB645_D8_usingintergenic_mappingfile > 18]
        .reset_index()
        .melt(id_vars=['barcode', 'new_locus_tag',], var_name='sample_id')
       .set_index('sample_id'))

In [ ]:
test = test.join(metadata)

In [ ]:
test['time'] = test['day'].map({'D1': 1,
                               'D2': 2,
                               'D3': 3,
                               'D4': 4, 
                               'D8': 8, 
                               'inoculum': 0})

In [ ]:
test

In [ ]:
px.line(test, x='time', y='value', color='mouse' )